# AirSense-R: Do Air-Quality Forecasters Keep Their Rank Under Sensor Degradation and Site Shift?

## 08b Independent saved-prediction verification

**Permitted partitions: FINAL TEST — READ ONLY.**

Use a CPU runtime and Run all. Mount the same Drive account and enable the AIRSENSE_GITHUB_TOKEN Colab secret. No SHA or stage entry is needed. This checks the 261 saved prediction bundles, recomputes metrics and paired uncertainty, and publishes separate review checkpoints. It does not train models, run inference or change experiment outputs. Drive reads and 2,000-replicate bootstrap calculations can take time; progress is printed per file batch and condition. After disconnection rerun cell 3 to reconnect; after runtime replacement use Run all. Completed condition reviews resume after all saved file hashes are rechecked. Never delete a lock. Save this notebook with outputs when finished.

## 1. Clone and verify source

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/Engr-Daniel/airsense-r.git"
# Use the full commit SHA of the GitHub revision containing this notebook.
# Resume using the SAME revision and run ID as the original execution.
TEST_MODE = os.environ.get("AIRSENSE_NOTEBOOK_TEST_MODE") == "1"
if TEST_MODE:
    ROOT = Path(os.environ["AIRSENSE_NOTEBOOK_TEST_ROOT"]).resolve()
else:
    CODE_REVISION = "c8e0b329fe2978f35b38054b0e0e409148cbe341"
    if len(CODE_REVISION) != 40 or any(c not in "0123456789abcdef" for c in CODE_REVISION.lower()):
        raise ValueError("Use a full Git commit SHA, not a moving branch name")
    ROOT = Path.cwd() / "airsense-r-p9-review-c8e0b329fe29"
    if not ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", CODE_REVISION], check=True)
    else:
        remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
        head = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
        dirty = subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()
        problems = []
        if remote != REPOSITORY:
            problems.append("origin is not the expected repository")
        if head != CODE_REVISION:
            problems.append(f"checkout is {head}, requested {CODE_REVISION}")
        if dirty:
            problems.append("checkout has modified/untracked files")
        if problems:
            raise RuntimeError("; ".join(problems) + ". Preserve existing work and use a fresh runtime filesystem; restarting only the kernel does not remove the checkout.")
    if sys.version_info < (3, 11):
        raise RuntimeError("Python 3.11+ required")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT) + "[notebooks]"], check=True)
    print("Code revision:", CODE_REVISION)
# Editable installs may not activate their .pth file in an already-running kernel.
# Activate only this checkout, and reject cached modules from another checkout.
import importlib
SOURCE_DIR = (ROOT / "src").resolve()
PACKAGE_DIR = SOURCE_DIR / "airsense_r"
if not (PACKAGE_DIR / "__init__.py").is_file():
    raise RuntimeError("Selected checkout has no airsense_r package")
for module_name, module in tuple(sys.modules.items()):
    if module_name == "airsense_r" or module_name.startswith("airsense_r."):
        location = getattr(module, "__file__", None)
        if location is None or not Path(location).resolve().is_relative_to(PACKAGE_DIR):
            raise RuntimeError("Another airsense_r checkout is already imported; restart the runtime")
source_path = str(SOURCE_DIR)
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
importlib.invalidate_caches()
import airsense_r
if Path(airsense_r.__file__).resolve() != PACKAGE_DIR / "__init__.py":
    raise RuntimeError("Imported package does not match the selected checkout")
print("Workspace:", ROOT)
print("Verified package:", airsense_r.__file__)

from airsense_r.artifacts import source_hash
if source_hash(ROOT) != "2a1be64aba6a5223bfdef61b54996ad90509397246e80947b8f58a080c02e8c5":
    raise RuntimeError("Notebook/source mismatch; reopen the pinned notebook")


## 2. Connect storage

In [ ]:
from airsense_r.colab import configure_github_secret
RUN_ID = "p9-target-history-review-01"
if TEST_MODE:
    MODEL_DIRECTORY = Path(os.environ["AIRSENSE_P9_REVIEW_TEST_MODELS"])
    RUN_ID = os.environ["AIRSENSE_P9_REVIEW_TEST_RUN"]
    REMOTE = "fixture"
else:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    MODEL_DIRECTORY = Path("/content/drive/MyDrive/AirSense-R/model-checkpoints")
    configure_github_secret(ROOT, userdata.get("AIRSENSE_GITHUB_TOKEN"))
    REMOTE = REPOSITORY
if not (MODEL_DIRECTORY / "storage.json").is_file():
    raise FileNotFoundError("Original model storage missing; check the Drive account")
JOB_DIRECTORY = ROOT / ".tmp/notebook-jobs" / RUN_ID
print("Ready: verify saved predictions and uncertainty. No training or inference.")


## 3. Verify saved predictions and publish review checkpoints

In [ ]:
from airsense_r.notebook_job import run_and_watch
command = [sys.executable, str(ROOT / "scripts/verify_target_history.py"),
           "--model-directory", str(MODEL_DIRECTORY), "--remote", REMOTE]
if TEST_MODE:
    import json
    command = json.loads(os.environ["AIRSENSE_P9_REVIEW_TEST_COMMAND"])
run_and_watch(command, JOB_DIRECTORY, ROOT)


## 4. Inspect completion

In [ ]:
import json
from airsense_r.artifacts import RunStore, source_hash
from airsense_r.notebook_job import status
if status(JOB_DIRECTORY)["state"] != "complete":
    raise RuntimeError("Review incomplete; rerun cell 3 to reconnect or inspect its error")
directory = ROOT / ".tmp/checkpoint-checkouts" / RUN_ID / "results" / RUN_ID
manifest = json.loads((directory / "manifest.json").read_bytes())
provenance = manifest["provenance"]
if provenance["verifier_source_hash"] != source_hash(ROOT):
    raise RuntimeError("Verifier provenance mismatch")
store = RunStore(directory, provenance)
completion = store.load_json("complete.json")
if not completion["passed"]:
    raise RuntimeError("Verification did not pass")
display(completion)
print("VERIFICATION COMPLETE. Save this notebook with outputs for final review.")
